# 5. 모델 유창성 평가 (LLM)

## 개요

이 노트북은 대규모 언어모델(LLM)의 **유창성(Fluency)**을 평가합니다.

### 평가 기준: Fluency Score

**유창성(Fluency)**은 생성된 텍스트가 얼마나 자연스럽고 문법적으로 올바른지 측정하는 지표입니다.

#### 평가 기준

1. **문법적 정확성 (Grammatical Correctness)**
   - 문장 구조가 올바른가?
   - 시제, 주어-동사 일치가 맞는가?

2. **자연스러움 (Naturalness)**
   - 사람이 쓴 것처럼 자연스러운가?
   - 어색한 표현이나 반복이 없는가?

3. **일관성 (Coherence)**
   - 문장 간 논리적 흐름이 있는가?
   - 문맥이 일관되는가?

#### 평가 척도: Likert Scale (1-5)

| 점수 | 수준 | 설명 |
|------|------|------|
| **5** | Excellent | 완벽한 문법, 매우 자연스러움 |
| **4** | Good | 사소한 문제, 전반적으로 자연스러움 |
| **3** | Fair | 명백한 오류, 어느 정도 이해 가능 |
| **2** | Poor | 여러 오류, 이해하기 어려움 |
| **1** | Very Poor | 심각한 오류, 거의 이해 불가 |

### 평가 방법: Self-Judge (LLM-as-a-Judge)

- **생성 모델**: DistilGPT-2 (작고 빠른 모델)
- **평가 모델**: GPT-2 또는 다른 LLM (Self-Judge)
- **프롬프트 기반 평가**: 평가 기준을 명시한 프롬프트로 점수 산출

### 데이터
- 다양한 프롬프트에서 텍스트 생성
- 각 생성물에 대해 유창성 점수 계산
- 평균 유창성 점수 산출

## 1. 환경 설정 및 모델 로드

In [ ]:
# 필요한 라이브러리 설치
!pip install -q transformers torch pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, pipeline
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 시각화 스타일
sns.set_style('whitegrid')
sns.set_palette('Set2')

# 디바이스 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {device}")

print("라이브러리 로드 완료")

In [ ]:
# 텍스트 생성 모델 로드 (DistilGPT-2)
print("DistilGPT-2 모델 로딩 중...")
generator_model_name = "distilgpt2"
generator_tokenizer = GPT2Tokenizer.from_pretrained(generator_model_name)
generator_model = GPT2LMHeadModel.from_pretrained(generator_model_name).to(device)

# Padding token 설정
generator_tokenizer.pad_token = generator_tokenizer.eos_token

print("✓ 생성 모델 로드 완료")

In [ ]:
# 평가 모델 로드 (GPT-2 - Self-Judge용)
print("GPT-2 평가 모델 로딩 중...")
judge_model_name = "gpt2"
judge_tokenizer = GPT2Tokenizer.from_pretrained(judge_model_name)
judge_model = GPT2LMHeadModel.from_pretrained(judge_model_name).to(device)

# Padding token 설정
judge_tokenizer.pad_token = judge_tokenizer.eos_token

print("✓ 평가 모델 로드 완료")

## 2. 텍스트 생성

In [ ]:
# 테스트용 프롬프트 정의
test_prompts = [
    "The future of artificial intelligence is",
    "Once upon a time in a small village",
    "Climate change is one of the most pressing issues",
    "The scientist discovered a new element that",
    "In the world of technology, innovation",
    "The best way to learn programming is",
    "During the summer vacation, we decided to",
    "The ancient civilization was known for",
    "Artificial neural networks work by",
    "The healthcare system needs to improve"
]

print(f"총 {len(test_prompts)}개의 프롬프트 준비 완료")

In [ ]:
def generate_text(prompt, max_length=50, num_return_sequences=1, temperature=0.8, top_p=0.9):
    """
    DistilGPT-2로 텍스트 생성
    
    Args:
        prompt: 입력 프롬프트
        max_length: 최대 생성 길이
        num_return_sequences: 생성할 시퀀스 수
        temperature: 샘플링 온도 (높을수록 다양함)
        top_p: Nucleus sampling 파라미터
    
    Returns:
        생성된 텍스트 리스트
    """
    inputs = generator_tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    outputs = generator_model.generate(
        inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        pad_token_id=generator_tokenizer.eos_token_id
    )
    
    generated_texts = [generator_tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return generated_texts

# 모든 프롬프트에서 텍스트 생성
print("텍스트 생성 중...")
generated_samples = []

for i, prompt in enumerate(test_prompts):
    print(f"\n[{i+1}/{len(test_prompts)}] Prompt: {prompt}")
    generated = generate_text(prompt, max_length=60, num_return_sequences=1)[0]
    
    # 프롬프트 이후 생성된 부분만 추출
    generated_only = generated[len(prompt):].strip()
    
    generated_samples.append({
        'prompt': prompt,
        'generated_full': generated,
        'generated_only': generated_only
    })
    
    print(f"Generated: {generated}")

print(f"\n✓ 총 {len(generated_samples)}개의 텍스트 생성 완료")

## 3. 유창성 평가 - Perplexity 기반

In [ ]:
def calculate_perplexity(text, model, tokenizer):
    """
    Perplexity 계산 (낮을수록 유창함)
    
    Args:
        text: 평가할 텍스트
        model: 언어 모델
        tokenizer: 토크나이저
    
    Returns:
        perplexity: Perplexity 값
    """
    encodings = tokenizer(text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**encodings, labels=encodings["input_ids"])
        loss = outputs.loss
    
    perplexity = torch.exp(loss).item()
    return perplexity

# Perplexity 계산
print("Perplexity 계산 중...")

for sample in generated_samples:
    text = sample['generated_full']
    ppl = calculate_perplexity(text, judge_model, judge_tokenizer)
    sample['perplexity'] = ppl

# 결과 출력
print("\n" + "="*100)
print("Perplexity 계산 결과 (낮을수록 유창함)")
print("="*100)
for i, sample in enumerate(generated_samples, 1):
    print(f"\n[{i}] Prompt: {sample['prompt']}")
    print(f"Generated: {sample['generated_full']}")
    print(f"Perplexity: {sample['perplexity']:.2f}")

avg_perplexity = np.mean([s['perplexity'] for s in generated_samples])
print(f"\n평균 Perplexity: {avg_perplexity:.2f}")
print("="*100)

## 4. 유창성 평가 - Rule-based Scoring

In [ ]:
def rule_based_fluency_score(text):
    """
    규칙 기반 유창성 점수 계산 (1-5 척도)
    
    평가 기준:
    - 문장 길이의 적절성
    - 반복 단어 비율
    - 문장 부호 사용
    - 단어 다양성
    
    Args:
        text: 평가할 텍스트
    
    Returns:
        score: 1-5 점수
        details: 평가 상세 정보
    """
    score = 5.0
    details = []
    
    words = text.split()
    num_words = len(words)
    
    # 1. 문장 길이 체크
    if num_words < 5:
        score -= 1.5
        details.append("Too short")
    elif num_words > 100:
        score -= 0.5
        details.append("Too long")
    
    # 2. 반복 단어 비율
    unique_words = len(set(words))
    repetition_ratio = 1 - (unique_words / num_words) if num_words > 0 else 0
    
    if repetition_ratio > 0.3:
        score -= 1.0
        details.append(f"High repetition ({repetition_ratio:.2f})")
    elif repetition_ratio > 0.2:
        score -= 0.5
        details.append(f"Some repetition ({repetition_ratio:.2f})")
    
    # 3. 연속 중복 단어
    consecutive_duplicates = sum(1 for i in range(len(words)-1) if words[i] == words[i+1])
    if consecutive_duplicates > 2:
        score -= 1.0
        details.append(f"Consecutive duplicates ({consecutive_duplicates})")
    elif consecutive_duplicates > 0:
        score -= 0.3
        details.append(f"Some duplicates ({consecutive_duplicates})")
    
    # 4. 문장 부호 체크 (완전한 문장인지)
    has_ending = text.strip().endswith(('.', '!', '?'))
    if not has_ending:
        score -= 0.5
        details.append("Incomplete sentence")
    
    # 5. 매우 짧은 단어들만 있는지 (평균 단어 길이)
    avg_word_length = np.mean([len(w) for w in words]) if words else 0
    if avg_word_length < 3:
        score -= 0.5
        details.append("Very short words")
    
    # 점수 범위 제한 (1-5)
    score = max(1.0, min(5.0, score))
    
    return score, details

# 규칙 기반 평가
print("규칙 기반 유창성 평가 중...")

for sample in generated_samples:
    text = sample['generated_full']
    score, details = rule_based_fluency_score(text)
    sample['fluency_score'] = score
    sample['score_details'] = details

# 결과 출력
print("\n" + "="*100)
print("규칙 기반 유창성 점수 (1-5 척도)")
print("="*100)
for i, sample in enumerate(generated_samples, 1):
    print(f"\n[{i}] Prompt: {sample['prompt']}")
    print(f"Generated: {sample['generated_full']}")
    print(f"Fluency Score: {sample['fluency_score']:.2f} / 5.0")
    if sample['score_details']:
        print(f"Issues: {', '.join(sample['score_details'])}")
    else:
        print("Issues: None (Excellent)")

avg_fluency = np.mean([s['fluency_score'] for s in generated_samples])
print(f"\n평균 유창성 점수: {avg_fluency:.2f} / 5.0")
print("="*100)

## 5. Perplexity를 유창성 점수로 변환

In [ ]:
def perplexity_to_score(perplexity):
    """
    Perplexity를 1-5 유창성 점수로 변환
    
    경험적 기준:
    - PPL < 50: Excellent (5)
    - PPL 50-100: Good (4)
    - PPL 100-200: Fair (3)
    - PPL 200-400: Poor (2)
    - PPL > 400: Very Poor (1)
    """
    if perplexity < 50:
        return 5.0
    elif perplexity < 100:
        return 4.0 + (100 - perplexity) / 50
    elif perplexity < 200:
        return 3.0 + (200 - perplexity) / 100
    elif perplexity < 400:
        return 2.0 + (400 - perplexity) / 200
    else:
        return max(1.0, 2.0 - (perplexity - 400) / 400)

# Perplexity 기반 점수 추가
for sample in generated_samples:
    sample['ppl_based_score'] = perplexity_to_score(sample['perplexity'])

# 종합 점수 (규칙 기반 + Perplexity 기반 평균)
for sample in generated_samples:
    sample['combined_score'] = (sample['fluency_score'] + sample['ppl_based_score']) / 2

# 결과 출력
print("\n" + "="*100)
print("종합 유창성 평가 결과")
print("="*100)
for i, sample in enumerate(generated_samples, 1):
    print(f"\n[{i}] {sample['prompt']}")
    print(f"Generated: {sample['generated_full']}")
    print(f"  - Rule-based Score: {sample['fluency_score']:.2f}")
    print(f"  - Perplexity-based Score: {sample['ppl_based_score']:.2f} (PPL: {sample['perplexity']:.2f})")
    print(f"  - Combined Score: {sample['combined_score']:.2f} / 5.0")

avg_combined = np.mean([s['combined_score'] for s in generated_samples])
print(f"\n평균 종합 유창성 점수: {avg_combined:.2f} / 5.0")
print("="*100)

## 6. 결과 시각화

In [ ]:
# DataFrame 생성
df_results = pd.DataFrame(generated_samples)

# 시각화 1: 점수 분포
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Rule-based Score 분포
axes[0, 0].hist(df_results['fluency_score'], bins=10, color='#3498db', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(df_results['fluency_score'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df_results['fluency_score'].mean():.2f}")
axes[0, 0].set_xlabel('Fluency Score', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Rule-based Fluency Score Distribution', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Perplexity 분포
axes[0, 1].hist(df_results['perplexity'], bins=10, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[0, 1].axvline(df_results['perplexity'].mean(), color='blue', linestyle='--', linewidth=2, label=f"Mean: {df_results['perplexity'].mean():.2f}")
axes[0, 1].set_xlabel('Perplexity', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Perplexity Distribution (Lower is Better)', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Perplexity vs Rule-based Score
axes[1, 0].scatter(df_results['perplexity'], df_results['fluency_score'], s=100, alpha=0.6, color='#2ecc71', edgecolor='black')
axes[1, 0].set_xlabel('Perplexity', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Rule-based Fluency Score', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Perplexity vs Rule-based Score', fontsize=13, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Combined Score
axes[1, 1].hist(df_results['combined_score'], bins=10, color='#9b59b6', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(df_results['combined_score'].mean(), color='orange', linestyle='--', linewidth=2, label=f"Mean: {df_results['combined_score'].mean():.2f}")
axes[1, 1].set_xlabel('Combined Fluency Score', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Combined Fluency Score Distribution', fontsize=13, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../assets/fluency_score_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 시각화 2: 개별 샘플 점수 비교
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(df_results))
width = 0.25

bars1 = ax.bar(x - width, df_results['fluency_score'], width, label='Rule-based', color='#3498db', alpha=0.8)
bars2 = ax.bar(x, df_results['ppl_based_score'], width, label='Perplexity-based', color='#e74c3c', alpha=0.8)
bars3 = ax.bar(x + width, df_results['combined_score'], width, label='Combined', color='#2ecc71', alpha=0.8)

ax.set_xlabel('Sample Index', fontsize=12, fontweight='bold')
ax.set_ylabel('Fluency Score (1-5)', fontsize=12, fontweight='bold')
ax.set_title('Fluency Scores Comparison by Sample', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f"S{i+1}" for i in range(len(df_results))])
ax.legend(loc='upper right')
ax.axhline(y=3.0, color='orange', linestyle='--', linewidth=1, alpha=0.5, label='Fair Threshold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../assets/fluency_scores_by_sample.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 시각화 3: 점수별 카테고리 분포
def categorize_score(score):
    if score >= 4.5:
        return 'Excellent (5)'
    elif score >= 3.5:
        return 'Good (4)'
    elif score >= 2.5:
        return 'Fair (3)'
    elif score >= 1.5:
        return 'Poor (2)'
    else:
        return 'Very Poor (1)'

df_results['score_category'] = df_results['combined_score'].apply(categorize_score)
category_counts = df_results['score_category'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#95a5a6']
category_order = ['Excellent (5)', 'Good (4)', 'Fair (3)', 'Poor (2)', 'Very Poor (1)']
ordered_counts = [category_counts.get(cat, 0) for cat in category_order]

wedges, texts, autotexts = ax.pie(ordered_counts, labels=category_order, autopct='%1.1f%%',
                                    colors=colors, startangle=90, explode=[0.05]*len(category_order))

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(11)

ax.set_title('Fluency Score Category Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/fluency_category_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. 상세 통계 요약

In [ ]:
# 종합 통계
summary_stats = pd.DataFrame({
    'Metric': ['Rule-based Score', 'Perplexity', 'Perplexity-based Score', 'Combined Score'],
    'Mean': [
        df_results['fluency_score'].mean(),
        df_results['perplexity'].mean(),
        df_results['ppl_based_score'].mean(),
        df_results['combined_score'].mean()
    ],
    'Std': [
        df_results['fluency_score'].std(),
        df_results['perplexity'].std(),
        df_results['ppl_based_score'].std(),
        df_results['combined_score'].std()
    ],
    'Min': [
        df_results['fluency_score'].min(),
        df_results['perplexity'].min(),
        df_results['ppl_based_score'].min(),
        df_results['combined_score'].min()
    ],
    'Max': [
        df_results['fluency_score'].max(),
        df_results['perplexity'].max(),
        df_results['ppl_based_score'].max(),
        df_results['combined_score'].max()
    ]
}).round(2)

print("\n" + "="*100)
print("유창성 평가 종합 통계")
print("="*100)
print(summary_stats.to_string(index=False))
print("="*100)

# 카테고리별 분포
print("\n점수 카테고리 분포:")
print(category_counts.to_string())

# 최고/최저 점수 샘플
best_idx = df_results['combined_score'].idxmax()
worst_idx = df_results['combined_score'].idxmin()

print("\n" + "="*100)
print("최고 유창성 샘플")
print("="*100)
print(f"Prompt: {df_results.loc[best_idx, 'prompt']}")
print(f"Generated: {df_results.loc[best_idx, 'generated_full']}")
print(f"Combined Score: {df_results.loc[best_idx, 'combined_score']:.2f}")

print("\n" + "="*100)
print("최저 유창성 샘플")
print("="*100)
print(f"Prompt: {df_results.loc[worst_idx, 'prompt']}")
print(f"Generated: {df_results.loc[worst_idx, 'generated_full']}")
print(f"Combined Score: {df_results.loc[worst_idx, 'combined_score']:.2f}")
print(f"Issues: {', '.join(df_results.loc[worst_idx, 'score_details']) if df_results.loc[worst_idx, 'score_details'] else 'None'}")
print("="*100)

## 8. 결론 및 개선 방향

### 평가 결과 해석

#### 유창성 점수 기준

| 점수 범위 | 수준 | 의미 |
|-----------|------|------|
| **4.5 ~ 5.0** | Excellent | 완벽한 문법, 매우 자연스러운 표현 |
| **3.5 ~ 4.5** | Good | 사소한 문제, 전반적으로 양호 |
| **2.5 ~ 3.5** | Fair | 명백한 오류, 이해는 가능 |
| **1.5 ~ 2.5** | Poor | 여러 오류, 이해하기 어려움 |
| **< 1.5** | Very Poor | 심각한 오류, 거의 이해 불가 |

#### Perplexity 기준

- **낮은 Perplexity (< 100)**: 모델이 텍스트를 "예상 가능"하다고 판단 → 일반적으로 유창함
- **높은 Perplexity (> 200)**: 모델이 텍스트를 "예상 불가능"하다고 판단 → 비정상적이거나 오류 많음

**주의**: Perplexity는 문법적 정확성보다는 "일반성"을 측정. 창의적이고 독특한 표현도 높은 PPL을 가질 수 있음.

### 유창성 저하의 주요 원인

#### 1. 반복 (Repetition)
- **문제**: 동일한 단어/구절 반복
- **예시**: "the the", "very very very good"
- **원인**: 모델의 temperature 설정, 학습 데이터 품질

#### 2. 불완전한 문장 (Incomplete Sentences)
- **문제**: 문장이 중간에 끊김
- **예시**: "The scientist discovered that"
- **원인**: max_length 제한, 모델 크기 부족

#### 3. 문법 오류 (Grammatical Errors)
- **문제**: 주어-동사 불일치, 시제 오류
- **예시**: "He are going", "She goed"
- **원인**: 모델 학습 부족, 데이터 품질

#### 4. 비논리적 내용 (Incoherence)
- **문제**: 앞뒤 문맥이 맞지 않음
- **예시**: "The cat flew to the moon and ate pizza"
- **원인**: 작은 모델 크기, 짧은 context window

### 개선 방향

#### 1. 생성 파라미터 튜닝

**Temperature 조정**
```python
# 낮은 temperature (0.5-0.7) → 더 안전하고 유창한 출력
generate_text(prompt, temperature=0.6)

# 높은 temperature (1.0-1.5) → 더 창의적이지만 오류 증가
generate_text(prompt, temperature=1.2)
```

**Top-p (Nucleus Sampling) 조정**
```python
# 높은 top_p (0.9-0.95) → 다양성 증가
generate_text(prompt, top_p=0.95)

# 낮은 top_p (0.7-0.8) → 안정성 증가
generate_text(prompt, top_p=0.75)
```

**Repetition Penalty 추가**
```python
outputs = model.generate(
    inputs,
    repetition_penalty=1.2,  # 반복 억제
    no_repeat_ngram_size=3   # 3-gram 반복 금지
)
```

#### 2. 모델 업그레이드

**더 큰 모델 사용**
- DistilGPT-2 (82M) → GPT-2 (124M) → GPT-2 Large (774M)
- 더 큰 모델일수록 유창성과 일관성 향상

**Fine-tuning**
```python
from transformers import Trainer, TrainingArguments

# 고품질 데이터로 fine-tuning
training_args = TrainingArguments(
    output_dir="./fluent-gpt2",
    num_train_epochs=3,
    per_device_train_batch_size=4
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=high_quality_dataset
)

trainer.train()
```

**RLHF (Reinforcement Learning from Human Feedback)**
- 사람의 선호도를 반영하여 유창성 개선
- ChatGPT, Claude 등의 핵심 기술

#### 3. 후처리 (Post-processing)

**반복 제거**
```python
def remove_repetitions(text):
    words = text.split()
    cleaned = []
    for i, word in enumerate(words):
        if i == 0 or word != words[i-1]:
            cleaned.append(word)
    return ' '.join(cleaned)
```

**문법 교정**
```python
from language_tool_python import LanguageTool

tool = LanguageTool('en-US')
corrected = tool.correct(generated_text)
```

#### 4. 평가 방법 개선

**Human Evaluation**
- 실제 사람이 유창성 평가 (Gold Standard)
- Amazon Mechanical Turk, Labelbox 등 활용

**GPT-4 as a Judge**
```python
from openai import OpenAI

client = OpenAI()

def gpt4_evaluate_fluency(text):
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "Rate the fluency of the following text on a scale of 1-5."},
            {"role": "user", "content": text}
        ]
    )
    return response.choices[0].message.content
```

**BERTScore, BARTScore**
- 학습된 언어 모델 기반 평가 메트릭
- 참조 텍스트 없이도 품질 평가 가능

```python
from bert_score import score

P, R, F1 = score([generated_text], [reference_text], lang="en")
```

### 유창성과 다른 품질 지표의 균형

- **유창성 vs 창의성**: 너무 안전한 출력은 지루할 수 있음
- **유창성 vs 사실성**: 문법적으로 완벽해도 틀린 내용일 수 있음
- **유창성 vs 다양성**: 모든 출력이 비슷하면 문제

→ **다면적 평가 필요**: 유창성, 관련성, 사실성, 다양성을 함께 측정

### 참고 자료
- "On the Evaluation of LLMs": https://arxiv.org/abs/2307.03109
- HuggingFace Generation: https://huggingface.co/docs/transformers/main_classes/text_generation
- Perplexity Explained: https://huggingface.co/docs/transformers/perplexity